# iSCORS-Net — Phase 2: Real Cell Training & Evaluation

**Standalone notebook — no dependency on Phase 1.**

**Workflow:**
1. **p2-setup** — Mount Drive, clone repo, install deps
2. **p2-config** — Set all paths and hyperparameters  ← *only cell you need to edit*
3. **p2-preprocess** — Load & preprocess real video (binning, flat-field, BG removal)
4. **p2-train** — Self-supervised training on real video (RECON_TAUS = 16..128)
5. **p2-inference** — Full-frame inference with trained model
6. **p2-gt-compare** — Compare model vs iSCORS MATLAB GT (.mat)  ← *direct quality check*
7. **p2-r2map** — R² confidence map (per-pixel physics fit quality)
8. **p2-checkerboard** — Cross-validation vs traditional iSCORS (diagonal resampling)
9. **p2-gnorm** — G_norm curve: empirical vs theory vs synthetic reference
10. **p2-download** — ZIP all results and download to local


In [ ]:
# ── p2-setup: Mount Drive, Clone Repo, Install Dependencies ───────────────
from google.colab import drive
import os, subprocess

drive.mount('/content/drive', force_remount=False)

# ── Clone / update repo ───────────────────────────────────────────────────
REPO   = 'https://github.com/breezy90126/iscors-net.git'
BRANCH = 'claude/beautiful-volta-gFUot'
REPO_DIR = '/content/iscors-net'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned — pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    print(f'Cloning {REPO} ({BRANCH}) ...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# ── Install dependencies ──────────────────────────────────────────────────
subprocess.run(['pip', 'install', '-q', 'tifffile', 'scipy', 'tqdm'], check=True)
print('Setup complete.')


In [ ]:
# ── p2-config: All Paths & Hyperparameters ────────────────────────────────
import os

VERSION   = 'v4.5'          # must match the checkpoint in p2-train

# ── Drive paths ───────────────────────────────────────────────────────────
ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'   # raw video zip
EXTRACT_DIR = '/content/real_data'                                   # local extraction
SAVE_DIR    = '/content/drive/MyDrive/iscors_test'                  # Drive results dir
CKPT_DIR    = '/content/iscors-net/checkpoint'

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,    exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)

# ── File names inside zip ─────────────────────────────────────────────────
VIDEO_FNAME = 'COBRI_rarw_video.tif'
MAT_FNAME   = 'Output_iSCORS_map.mat'

# ── Preprocessing ─────────────────────────────────────────────────────────
N_FRAMES   = 2000        # frames to load
BIN_FACTOR = 2           # spatial binning (2→2×2, 4→4×4)
CHUNK_SIZE = 100         # frames per read chunk

# ── Training (real video) ─────────────────────────────────────────────────
# G(0)=CV² normalisation (v4.5): G_norm(τ)=G(τ)/G(0)=1/(1+γτ^α).
# G(0) is always the largest G value → stable for any diffusion speed.
# Matches MATLAB iSCORS 'nor_1'. γ is now fully identifiable.
RECON_TAUS  = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)   # K=10
G0_NORM     = True      # use G(τ=0)=CV² anchor (v4.5, matches MATLAB)
EPOCHS      = 150
BATCH_SIZE  = 4
LR          = 1e-4
PATCH_SIZE  = 64
TRAIN_FRAC  = 0.65

# ── σ_clip: remove normalization-unstable pixels ──────────────────────────
# With τ_ref=1, σ_G_norm(τ=1) is naturally small for all cell pixels
# (G(τ=1) is the largest G value), so sigma_clip is mostly a safety net.
# Set to None to disable.  2.0 is conservative and removes near nothing.
SIGMA_CLIP  = 2.0

# ── Training throughput cap ───────────────────────────────────────────────
# Caps patches per epoch so each epoch takes ~30–60s on a 15 GB GPU.
# 2000 patches / BATCH_SIZE=4 → 500 steps/epoch ≈ 30s → 150 epochs ≈ 75 min.
# Set to None to use all available patches (may take hours per epoch).
MAX_PATCHES_PER_EPOCH = 2000

# ── Optical parameters for physics-derived TV ─────────────────────────────
WAVELENGTH_NM = 532.0
NA            = 1.4
PIXEL_SIZE_NM = 65.0 * BIN_FACTOR          # binned pixel size

REAL_CKPT = os.path.join(CKPT_DIR, f'pissl_phys_recon_{VERSION}_real.pth')

print(f'VERSION               : {VERSION}')
print(f'RECON_TAUS            : {RECON_TAUS}  (τ_ref={RECON_TAUS[0]})')
print(f'G0_NORM               : {G0_NORM}')
print(f'SIGMA_CLIP            : {SIGMA_CLIP}')
print(f'MAX_PATCHES_PER_EPOCH : {MAX_PATCHES_PER_EPOCH}')
print(f'BIN_FACTOR            : {BIN_FACTOR}')
print(f'N_FRAMES              : {N_FRAMES}')
print(f'EPOCHS                : {EPOCHS}')
print(f'REAL_CKPT             : {REAL_CKPT}')
print(f'SAVE_DIR              : {SAVE_DIR}')


In [ ]:
# ── p2-preprocess: Load & Preprocess Real Video ───────────────────────────
# Pipeline: extract zip (cached) → chunked binning → flat-field → BG removal
import zipfile, os, numpy as np, tifffile, matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

def _find(root, name):
    for dp, _, fs in os.walk(root):
        if name in fs:
            return os.path.join(dp, name)
    return None

VIDEO_PATH = _find(EXTRACT_DIR, VIDEO_FNAME)
MAT_PATH   = _find(EXTRACT_DIR, MAT_FNAME)

if VIDEO_PATH and MAT_PATH:
    print('Files already extracted — skipping zip.')
else:
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    VIDEO_PATH = _find(EXTRACT_DIR, VIDEO_FNAME)
    MAT_PATH   = _find(EXTRACT_DIR, MAT_FNAME)

assert VIDEO_PATH, f'{VIDEO_FNAME} not found'
assert MAT_PATH,   f'{MAT_FNAME} not found'
print(f'Video : {VIDEO_PATH}')
print(f'MAT   : {MAT_PATH}')

# ── Frame shape & binning ─────────────────────────────────────────────────
frame0 = tifffile.imread(VIDEO_PATH, key=0).astype(np.float32)
H_orig, W_orig = frame0.shape
H_bin,  W_bin  = H_orig // BIN_FACTOR, W_orig // BIN_FACTOR
print(f'Frame: {H_orig}×{W_orig}  →  binned: {H_bin}×{W_bin}  ({BIN_FACTOR}×{BIN_FACTOR})')

with tifffile.TiffFile(VIDEO_PATH) as tf:
    total_frames = len(tf.pages)
n_frames = min(N_FRAMES, total_frames)
print(f'Loading {n_frames}/{total_frames} frames ...')

video_raw = np.empty((n_frames, H_bin, W_bin), dtype=np.float32)
for start in range(0, n_frames, CHUNK_SIZE):
    end   = min(start + CHUNK_SIZE, n_frames)
    chunk = tifffile.imread(VIDEO_PATH, key=range(start, end)).astype(np.float32)
    T_c   = end - start
    video_raw[start:end] = (chunk
        .reshape(T_c, H_bin, BIN_FACTOR, W_bin, BIN_FACTOR)
        .mean(axis=(2, 4)))
    del chunk
    if start % (CHUNK_SIZE * 5) == 0:
        print(f'  {start:4d}/{n_frames}')

T, H, W = video_raw.shape
print(f'Loaded: T={T}  H={H}  W={W}  range=[{video_raw.min():.1f}, {video_raw.max():.1f}]')

# Step 1: flat-field
print('Step 1: flat-field ...')
median_xy = np.median(video_raw, axis=0)
video_ff  = video_raw / (median_xy[np.newaxis] + 1e-10)

# Step 2: per-frame Gaussian BG removal
print('Step 2: Gaussian BG removal (sigma=4 binned px) ...')
video_proc = np.empty_like(video_ff)
for t in range(T):
    bg = gaussian_filter(video_ff[t], sigma=4)
    video_proc[t] = video_ff[t] / (bg + 1e-10)
    if t % max(1, T//5) == 0:
        print(f'  {t}/{T}')

print(f'Done: mean={video_proc.mean():.4f}  std={video_proc.std():.6f}')
print('>>> mean ≈ 1.0 → G(τ) formula valid ✓')

# ── Quick 3-panel diagnostic ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title, cmap in [
    (axes[0], video_raw[0],   f'Frame 0 raw ({H}×{W})',          'gray'),
    (axes[1], video_proc[0],  'Frame 0 processed (mean≈1)',       'gray'),
    (axes[2], video_proc[0]-1,'δI/I  (zero-centered fluctuation)','RdBu_r'),
]:
    p1, p99 = np.percentile(data, 1), np.percentile(data, 99)
    im = ax.imshow(data, cmap=cmap, vmin=p1, vmax=p99)
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=10);  ax.axis('off')
plt.tight_layout();  plt.show()


In [ ]:
# ── p2-train: Self-supervised Training on Real Video ─────────────────────
import sys, os, numpy as np, torch, torch.optim as optim, tqdm, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, RandomSampler
from scipy.optimize import curve_fit

for _k in list(sys.modules.keys()):
    if _k in ('datasets','models','loss') or _k.startswith(('datasets.','models.','loss.')):
        del sys.modules[_k]

from datasets.phys_recon_dataset import PhysReconDataset
from models.pissl_tau_encoder    import PISSLTauEncoder
from loss.phys_recon_loss        import PhysicsReconLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}  RECON_TAUS={RECON_TAUS}  G0_NORM={G0_NORM}  SIGMA_CLIP={SIGMA_CLIP}')

# ── Dataset ───────────────────────────────────────────────────────────────
print('Building dataset ...')
real_ds = PhysReconDataset(
    video_tensor=video_proc, recon_taus=RECON_TAUS,
    patch_size=PATCH_SIZE, mode='train', train_fraction=TRAIN_FRAC,
    sigma_clip=SIGMA_CLIP,
)

# ── DataLoader with per-epoch patch cap ───────────────────────────────────
if MAX_PATCHES_PER_EPOCH is not None and MAX_PATCHES_PER_EPOCH < len(real_ds):
    sampler = RandomSampler(real_ds, replacement=False, num_samples=MAX_PATCHES_PER_EPOCH)
    loader  = DataLoader(real_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    print(f'RandomSampler: {MAX_PATCHES_PER_EPOCH}/{len(real_ds)} patches/epoch '
          f'→ {MAX_PATCHES_PER_EPOCH//BATCH_SIZE} steps/epoch')
else:
    loader = DataLoader(real_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    print(f'Full dataset: {len(real_ds)} patches → {len(loader)} steps/epoch')

# ── Auto-estimate Fisher prior from mean G_norm curve ─────────────────────
tau_ref_f  = float(RECON_TAUS[0])
tau_arr_f  = np.array(RECON_TAUS, dtype=np.float32)
g_mean_fit = real_ds.g_norm[real_ds.cell_mask].mean(axis=0)   # (K,)

def _g_norm_theory(tau, gamma, alpha):
    if G0_NORM:
        return 1.0 / (1.0 + gamma * tau**alpha)
    return (1 + gamma * tau_ref_f**alpha) / (1 + gamma * tau**alpha)

try:
    popt, _ = curve_fit(_g_norm_theory, tau_arr_f, g_mean_fit,
                        p0=[0.05, 0.8], bounds=([1e-4, 0.1], [1.0, 2.0]),
                        maxfev=3000)
    F_G0, F_A0 = float(popt[0]), float(popt[1])
    print(f'Fisher prior (auto-fit): γ₀={F_G0:.4f}  α₀={F_A0:.4f}')
except Exception as e:
    F_G0, F_A0 = 0.05, 0.8
    print(f'Prior fit failed ({e}) — using defaults: γ₀={F_G0}  α₀={F_A0}')

# ── Model & loss ──────────────────────────────────────────────────────────
model = PISSLTauEncoder(recon_taus=RECON_TAUS, predict_amplitude=False).to(device)
crit  = PhysicsReconLoss(
    recon_taus=RECON_TAUS,
    g0_norm=G0_NORM,             # v4.5: G(τ)/G(0) target
    shape_only=(not G0_NORM),    # legacy fallback if G0_NORM=False
    fisher_weighted=True, fisher_gamma_prior=F_G0, fisher_alpha_prior=F_A0,
).to(device)
opt   = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

# Physics-derived TV lambdas
L_psf   = 0.61 * WAVELENGTH_NM / NA / PIXEL_SIZE_NM
sigma2G = 1.0 / T
lam_g   = sigma2G / ((1.0/6)**2 * L_psf**2) * 10.0
lam_a   = sigma2G / ((2.0/6)**2 * L_psf**2) * 10.0

# v4.6 — adaptive TV: weight the smoothing strength by *un*-reliability so
# that noisy / model-misspecified pixels get smoothed harder while clean,
# well-fit pixels keep their detail. The weight map is normalised to ≈1 on
# average over cell pixels, so the overall λ_TV scale stays comparable to
# v4.5 — only the *spatial distribution* of smoothing changes.
#   noise : σ_G_norm (mean over τ)         — measurement reliability
#   resid : |G_theory(γ̂,α̂) − G_empirical|² — does a single power law
#           explain THIS pixel's curve? High inside multi-component
#           (nucleus) regions where the physics model is misspecified
#           (model misfit ≠ measurement noise — both reduce trust in TV).
taus_t = torch.tensor(RECON_TAUS, dtype=torch.float32, device=device)  # (K,)

def _mhtv_adaptive(x, cm, w, delta=0.05):
    m  = cm.unsqueeze(1).float()
    wx = 0.5*(w[...,1:]+w[...,:-1]);  wy = 0.5*(w[...,1:,:]+w[...,:-1,:])
    dx = x[...,1:]-x[...,:-1];        dy = x[...,1:,:]-x[...,:-1,:]
    mx = m[...,1:]*m[...,:-1];        my = m[...,1:,:]*m[...,:-1,:]
    def _h(t): a=t.abs(); return torch.where(a<delta,0.5*t**2/delta,a-0.5*delta)
    n = (mx*wx).sum()+(my*wy).sum()+1e-10
    return (_h(dx)*mx*wx).sum()/n + (_h(dy)*my*wy).sum()/n

fw = crit.fisher_weights.squeeze().cpu().tolist()
print('Fisher weights: ' + '  '.join(f'τ={t}:{w:.3f}' for t,w in zip(RECON_TAUS,fw)))
print(f'λ_TV_γ={lam_g:.3e}  λ_TV_α={lam_a*3:.3e}  PSF={L_psf:.2f}px  (adaptive weighting v4.6)')

# ── Training loop ─────────────────────────────────────────────────────────
print(f'\nTraining {EPOCHS} epochs ...')
hist = {'loss':[], 'phys':[]}
model.train()
for epoch in range(EPOCHS):
    el=ep=gs=as_=px=0.0
    pb = tqdm.tqdm(loader, desc=f'Ep {epoch+1}/{EPOCHS}', leave=False)
    for g_in,g_tgt,mask,cell_p,sigma_p in pb:
        g_in=g_in.to(device); g_tgt=g_tgt.to(device)
        mask=mask.to(device); cell_p=cell_p.to(device); sigma_p=sigma_p.to(device)
        opt.zero_grad()
        pred = model(g_in)
        pl   = crit(pred, g_tgt, mask, sigma_g_norm=sigma_p)

        # Adaptive reliability weight (v4.6). Detached from the autograd graph
        # — using the model's own current (γ̂,α̂) as a proxy for "is a single
        # power law adequate here" must not create a gradient feedback loop
        # between the TV penalty and the very prediction it derives from.
        with torch.no_grad():
            g_hat = pred[:, 0, :, :]                                        # (B,P,P)
            a_hat = pred[:, 1, :, :]
            g_th  = 1.0 / (1.0 + g_hat.unsqueeze(-1) * taus_t.pow(a_hat.unsqueeze(-1)))
            resid = (g_th - g_tgt).pow(2).mean(dim=-1)                      # model misfit
            noise = sigma_p.mean(dim=-1)                                    # measurement noise
            raw_w = noise + resid
            cell_b = cell_p.bool()
            w_ref  = raw_w[cell_b].mean() if cell_b.any() else raw_w.mean()
            w_map  = (raw_w / (w_ref + 1e-10)).clamp(0.2, 5.0).unsqueeze(1)  # (B,1,P,P)

        tvg  = _mhtv_adaptive(pred[:,0:1], cell_p, w_map)
        tva  = _mhtv_adaptive(pred[:,1:2], cell_p, w_map)
        loss = pl + lam_g*tvg + lam_a*3.0*tva
        loss.backward();  opt.step()
        el+=loss.item();  ep+=pl.item()
        with torch.no_grad():
            m1=mask.unsqueeze(1)
            gs+=(pred[:,0:1]*m1).sum().item(); as_+=(pred[:,1:2]*m1).sum().item()
            px+=m1.sum().item()
        pb.set_postfix({'L':f'{loss.item():.4e}',
                        'γ':f'{gs/(px+1e-9):.3f}','α':f'{as_/(px+1e-9):.3f}'})
    n=max(len(loader),1);  sched.step()
    hist['loss'].append(el/n);  hist['phys'].append(ep/n)
    if (epoch+1)%30==0:
        print(f'  Ep {epoch+1}  Loss={el/n:.4e}  γ={gs/px:.3f}  α={as_/px:.3f}')

torch.save(model.state_dict(), REAL_CKPT)
print(f'\nCheckpoint → {REAL_CKPT}')

fig_l, ax_l = plt.subplots(figsize=(7,3))
ax_l.plot(hist['loss'], label='Total');  ax_l.plot(hist['phys'],'--',label='Physics')
ax_l.set_yscale('log');  ax_l.set_xlabel('Epoch');  ax_l.legend()
mode_str = 'G0_norm' if G0_NORM else f'τ_ref={RECON_TAUS[0]}'
ax_l.set_title(f'Real training [{VERSION}]  {mode_str}  γ₀={F_G0:.3f} α₀={F_A0:.3f}')
plt.tight_layout();  plt.show()
fig_l.savefig(os.path.join(SAVE_DIR, f'real_loss_{VERSION}.png'), dpi=120, bbox_inches='tight')


In [ ]:
# ── p2-inference: Full-frame Inference ────────────────────────────────────
import sys, os, numpy as np, torch, tifffile, matplotlib.pyplot as plt
import torch.nn.functional as F_pad

for _k in list(sys.modules.keys()):
    if _k in ('datasets','models') or _k.startswith(('datasets.','models.')):
        del sys.modules[_k]

from datasets.phys_recon_dataset import PhysReconDataset
from models.pissl_tau_encoder    import PISSLTauEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build inference dataset (mode=eval → full masked G_norm, no blind-spot)
print('Building inference dataset ...')
infer_ds = PhysReconDataset(
    video_tensor=video_proc, recon_taus=RECON_TAUS,
    patch_size=PATCH_SIZE, mode='eval',
)
cell_mask = infer_ds.cell_mask

# Load model
model_inf = PISSLTauEncoder(recon_taus=RECON_TAUS, predict_amplitude=False).to(device)
model_inf.load_state_dict(torch.load(REAL_CKPT, map_location=device))
model_inf.eval()
print(f'Loaded: {REAL_CKPT}')

# Pad to multiple of 8
full_inp = infer_ds[0]                                         # (K, H, W)
K_inf, Hv, Wv = full_inp.shape
ph = (8 - Hv % 8) % 8;  pw = (8 - Wv % 8) % 8
if ph or pw:
    full_inp = F_pad.pad(full_inp, (0, pw, 0, ph))
    print(f'Padded: {Hv}×{Wv} → {Hv+ph}×{Wv+pw}')

with torch.no_grad():
    preds_out = model_inf(full_inp.unsqueeze(0).to(device))

gamma_pred = preds_out[0,0].cpu().numpy()[:Hv, :Wv]
alpha_pred = preds_out[0,1].cpu().numpy()[:Hv, :Wv]
gamma_pred[~cell_mask] = np.nan
alpha_pred[~cell_mask] = np.nan

# Stats
gc = gamma_pred[cell_mask]; ac = alpha_pred[cell_mask]
print(f'\nCell pixels: {cell_mask.sum()}  ({100*cell_mask.mean():.1f}%)')
print(f'γ  mean={gc.mean():.4f}  std={gc.std():.4f}  p1={np.percentile(gc,1):.4f}  p99={np.percentile(gc,99):.4f}')
print(f'α  mean={ac.mean():.4f}  std={ac.std():.4f}  p1={np.percentile(ac,1):.4f}  p99={np.percentile(ac,99):.4f}')

# Plot
fig_inf, axes_inf = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, cmap, title in [
    (axes_inf[0], gamma_pred, 'magma',  f'Model γ  [{VERSION}]'),
    (axes_inf[1], alpha_pred, 'plasma', f'Model α  [{VERSION}]'),
]:
    p1=np.nanpercentile(data,1); p99=np.nanpercentile(data,99)
    im = ax.imshow(data, cmap=cmap, vmin=p1, vmax=p99)
    plt.colorbar(im, ax=ax, label=f'[{p1:.3f},{p99:.3f}]')
    ax.set_title(title, fontsize=12);  ax.axis('off')
plt.suptitle('Real Cell Inference  (1st–99th percentile, cell mask only)', fontsize=13)
plt.tight_layout();  plt.show()

# Save
tifffile.imwrite(os.path.join(SAVE_DIR, f'model_gamma_{VERSION}_real.tif'), gamma_pred.astype(np.float32))
tifffile.imwrite(os.path.join(SAVE_DIR, f'model_alpha_{VERSION}_real.tif'), alpha_pred.astype(np.float32))
fig_inf.savefig(os.path.join(SAVE_DIR, f'model_maps_{VERSION}_real.png'), dpi=120, bbox_inches='tight')
print(f'Saved → {SAVE_DIR}')

# Save G_norm channels
g_norm_np = infer_ds.g_norm[:Hv, :Wv]                        # (H, W, K)
for i, tau in enumerate(RECON_TAUS):
    tifffile.imwrite(os.path.join(SAVE_DIR, f'gnorm_ch{i:02d}_tau{tau}.tif'),
                     g_norm_np[:,:,i].astype(np.float32))
print(f'Saved {len(RECON_TAUS)} G_norm channel TIFs')


In [ ]:
# ── p2-gt-compare: Model vs iSCORS MATLAB GT (.mat) ────────────────────
# Loads the traditional iSCORS result from the .mat file and compares
# directly against model inference on the same video.  No spatial subsampling
# (unlike checkerboard which uses gridA/gridB at H/2×W/2).
import numpy as np, matplotlib.pyplot as plt, os, scipy.io
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import zoom

# ── 1. Load .mat GT ───────────────────────────────────────────────────────
print(f'Loading: {MAT_PATH}')
try:
    mat = scipy.io.loadmat(MAT_PATH)
    mat_keys = [k for k in mat.keys() if not k.startswith('_')]
    print(f'MATLAB format  ({len(mat_keys)} keys)')
except NotImplementedError:  # HDF5 / v7.3 .mat
    import h5py
    mat = {}
    with h5py.File(MAT_PATH, 'r') as hf:
        for k in hf.keys():
            mat[k] = np.array(hf[k])
    mat_keys = list(mat.keys())
    print(f'HDF5 format  ({len(mat_keys)} keys)')

print('Available arrays:')
for k in mat_keys:
    v = mat[k]
    if hasattr(v, 'shape') and v.ndim >= 2:
        print(f'  {k:25s} shape={str(v.shape):20s} '
              f'range=[{float(np.nanmin(v)):.4f}, {float(np.nanmax(v)):.4f}]')

# ── 2. Load GT γ field ───────────────────────────────────────────────────
# iSCORS MATLAB typically outputs:
#   D_map   — diffusion coefficient  → γ_ours ∝ 1/D  (set GT_GAMMA_IS_D=True)
#   Cond_map— chromatin condensation → NOT α (skip α comparison)
#   V_map   — active transport velocity (not used here)
#
# Set GT_GAMMA_KEY manually if auto-detect picks the wrong field.
# Set GT_GAMMA_IS_D=True when MATLAB stores D (diffusion coeff); the cell
#   will compare model γ against 1/D_map (since γ ∝ 1/D).
# Leave GT_ALPHA_KEY=None — iSCORS GT has no true α map.
GT_GAMMA_KEY   = None   # e.g. 'D_map'; None = auto-detect
GT_GAMMA_IS_D  = True   # True  → GT field is D; comparison uses 1/D vs our γ
                         # False → GT field is γ directly
GT_ALPHA_KEY   = None   # None  = skip α comparison (GT has no α)

GAMMA_KEYS = ['D_map','Dmap','D','gamma','Gamma','GAMMA',
              'gamma_map','GammaMap','diffusion','Diffusion']

gt_gamma_raw, gt_alpha_raw = None, None
for k in ([GT_GAMMA_KEY] if GT_GAMMA_KEY else GAMMA_KEYS):
    if k in mat and hasattr(mat[k],'shape') and mat[k].squeeze().ndim == 2:
        gt_gamma_raw = mat[k].astype(np.float64).squeeze()
        print(f'  γ GT ← mat["{k}"]  (GT_GAMMA_IS_D={GT_GAMMA_IS_D})')
        break

if gt_gamma_raw is None:
    print('\n⚠ γ auto-detect failed. Set GT_GAMMA_KEY to one of the keys printed above.')
    raise ValueError('Manual assignment needed')
if GT_ALPHA_KEY is None:
    print('  α GT: skipped (iSCORS GT has no true α — Cond_map is condensation, not α)')

# ── 3. Transpose if MATLAB column-major stored (shape H>W check) ──────────
Hm, Wm = gamma_pred.shape
for arr_name, arr in ([('gt_gamma_raw', gt_gamma_raw)]
                       + ([('gt_alpha_raw', gt_alpha_raw)] if gt_alpha_raw is not None else [])):
    Hg, Wg = arr.shape
    if abs(Hg - Wm) < abs(Hg - Hm) and Hg != Hm:
        print(f'  Transposing {arr_name}: {arr.shape} → ', end='')
        if arr_name == 'gt_gamma_raw': gt_gamma_raw = gt_gamma_raw.T
        else:                          gt_alpha_raw  = gt_alpha_raw.T
        print(gt_gamma_raw.shape if arr_name=='gt_gamma_raw' else gt_alpha_raw.shape)

# ── 4. Resolution alignment ───────────────────────────────────────────────
Hg, Wg = gt_gamma_raw.shape
def _align(arr):
    if arr is None: return None
    if arr.shape != (Hg, Wg): arr = arr  # already same shape
    if (arr.shape[0], arr.shape[1]) != (Hm, Wm):
        from scipy.ndimage import zoom as _z
        return _z(arr, (Hm/arr.shape[0], Wm/arr.shape[1]), order=1).astype(np.float32)
    return arr.astype(np.float32)

if (Hg, Wg) != (Hm, Wm):
    print(f'Resizing GT {Hg}×{Wg} → {Hm}×{Wm} (model resolution)')
gt_gamma_r = _align(gt_gamma_raw)
gt_alpha_r = _align(gt_alpha_raw)   # None if GT_ALPHA_KEY=None

# ── 4b. MATLAB → Python coordinate transform ─────────────────────────────
# MATLAB axis convention: flip vertically (上下反轉) then rotate 90° clockwise (右轉90°).
def _coord(arr):
    if arr is None: return None
    out = np.rot90(np.flipud(arr), k=-1).astype(np.float32)
    if out.shape != (Hm, Wm):
        out = zoom(out, (Hm/out.shape[0], Wm/out.shape[1]), order=1).astype(np.float32)
    return out
gt_gamma_r = _coord(gt_gamma_r)
gt_alpha_r = _coord(gt_alpha_r)
print(f'MATLAB→Python coord transform applied (flipud + rot90 clockwise)')

# ── 4c. Invert D → 1/D if MATLAB stores diffusion coefficient ────────────
if GT_GAMMA_IS_D:
    eps_d = np.nanpercentile(gt_gamma_r[gt_gamma_r > 0], 1) * 0.1 if (gt_gamma_r > 0).any() else 1e-6
    gt_gamma_r = 1.0 / (gt_gamma_r + eps_d)
    print(f'GT_GAMMA_IS_D: inverted D → 1/D  '
          f'range=[{np.nanmin(gt_gamma_r):.4f}, {np.nanmax(gt_gamma_r):.4f}]')

gt_gamma_r[~cell_mask] = np.nan
if gt_alpha_r is not None: gt_alpha_r[~cell_mask] = np.nan

# ── 5. Valid-pixel mask and scatter stats ─────────────────────────────────
ok_g = (cell_mask & np.isfinite(gt_gamma_r) & np.isfinite(gamma_pred) & (gt_gamma_r > 0))
gg = gt_gamma_r[ok_g].astype(float); mg = gamma_pred[ok_g].astype(float)
pg, _ = pearsonr(gg, mg);  sg = spearmanr(gg, mg).statistic;  eg = np.abs(mg-gg).mean()
print(f'\nValid γ pixels: {ok_g.sum()} / {cell_mask.sum()}')
print(f'\n=== Model vs iSCORS GT [{VERSION}] ===')
print(f'  γ  Pearson={pg:.3f}  Spearman={sg:.3f}  MAE={eg:.4f}'
      f'  GT_mean={gg.mean():.4f}  model_mean={mg.mean():.4f}')

has_alpha_gt = gt_alpha_r is not None
if has_alpha_gt:
    ok_a = (cell_mask & np.isfinite(gt_alpha_r) & np.isfinite(alpha_pred) & (gt_alpha_r > 0))
    ga = gt_alpha_r[ok_a].astype(float); ma = alpha_pred[ok_a].astype(float)
    pa, _ = pearsonr(ga, ma);  sa = spearmanr(ga, ma).statistic;  ea = np.abs(ma-ga).mean()
    print(f'  α  Pearson={pa:.3f}  Spearman={sa:.3f}  MAE={ea:.4f}'
          f'  GT_mean={ga.mean():.4f}  model_mean={ma.mean():.4f}')
else:
    print('  α  (no GT α field — set GT_ALPHA_KEY to compare)')
    pa = sa = ea = float('nan')

# ── 6. Spatial maps ───────────────────────────────────────────────────────
n_rows = 2 if has_alpha_gt else 1
fig_gt, axes_gt = plt.subplots(n_rows, 3, figsize=(15, 5*n_rows))
if n_rows == 1: axes_gt = axes_gt[np.newaxis, :]

rows_cfg = [(gt_gamma_r, gamma_pred, 'γ', 'magma', eg)]
if has_alpha_gt:
    rows_cfg.append((gt_alpha_r, alpha_pred, 'α', 'plasma', ea))

for row, (gt_map, md_map, lbl, cmap, mae_v) in enumerate(rows_cfg):
    diff = np.abs(md_map - gt_map)
    for col, (arr, ttl) in enumerate([
        (gt_map, f'{lbl} iSCORS GT  ({"1/D_map" if (lbl=="γ" and GT_GAMMA_IS_D) else "GT"})'),
        (md_map, f'{lbl} model [{VERSION}]'),
        (diff,   f'|Δ{lbl}|  MAE={mae_v:.4f}'),
    ]):
        p1  = np.nanpercentile(arr, 1)
        p99 = np.nanpercentile(arr, 99)
        im = axes_gt[row, col].imshow(
            arr, cmap='hot' if col==2 else cmap, vmin=p1, vmax=p99)
        plt.colorbar(im, ax=axes_gt[row,col], fraction=0.046, pad=0.04)
        axes_gt[row,col].set_title(ttl, fontsize=10)
        axes_gt[row,col].axis('off')

alpha_str = f'  α Pearson={pa:.3f}' if has_alpha_gt else '  α: no GT'
d_label = '1/D' if GT_GAMMA_IS_D else 'γ'
fig_gt.suptitle(f'Model vs iSCORS GT [{VERSION}]  γ vs {d_label}  Pearson={pg:.3f}{alpha_str}',
                fontsize=13)
plt.tight_layout();  plt.show()

# ── 7. Scatter plots ──────────────────────────────────────────────────────
n_sc = 2 if has_alpha_gt else 1
fig_sc2, sc_axes = plt.subplots(1, n_sc, figsize=(5*n_sc+1, 4))
if n_sc == 1: sc_axes = [sc_axes]

for ax, gt_v, md_v, pr, sr, mae, nm in (
    [(sc_axes[0], gg, mg, pg, sg, eg, 'γ')]
    + ([(sc_axes[1], ga, ma, pa, sa, ea, 'α')] if has_alpha_gt else [])
):
    ax.scatter(gt_v, md_v, alpha=0.05, s=1, c='steelblue')
    lim = [min(gt_v.min(), md_v.min()), max(gt_v.max(), md_v.max())]
    ax.plot(lim, lim, 'r--', lw=1, label='y=x');  ax.legend(fontsize=9)
    gt_label = f'1/D_map (.mat)' if (nm=='γ' and GT_GAMMA_IS_D) else f'{nm} iSCORS GT (.mat)'
    ax.set_xlabel(gt_label);  ax.set_ylabel(f'{nm} model [{VERSION}]')
    ax.set_title(f'{nm}: Pearson={pr:.3f}  Spearman={sr:.3f}  MAE={mae:.4f}')
fig_sc2.suptitle(f'Model vs iSCORS GT Scatter [{VERSION}]', fontsize=13)
plt.tight_layout();  plt.show()

fig_gt.savefig(os.path.join(SAVE_DIR, f'gt_compare_{VERSION}_real.png'),
               dpi=120, bbox_inches='tight')
fig_sc2.savefig(os.path.join(SAVE_DIR, f'gt_scatter_{VERSION}_real.png'),
                dpi=120, bbox_inches='tight')
print(f'Saved → {SAVE_DIR}')


In [ ]:
# ── p2-r2map: R² Confidence Map ──────────────────────────────────────────
# R²(y,x) = 1 - SS_res/SS_tot over τ dimension.
# Quantifies how well (γ_pred, α_pred) fits the empirical G_norm curve.
import numpy as np, tifffile, matplotlib.pyplot as plt, os

tau_arr_r2 = np.array(RECON_TAUS, dtype=np.float32)
tau_ref_r2 = float(RECON_TAUS[0])
eps        = 1e-10

gamma_3d = np.where(cell_mask, gamma_pred, 0.0)[:, :, None]
alpha_3d = np.where(cell_mask, alpha_pred, 1.0)[:, :, None]
tau_3d   = tau_arr_r2[None, None, :]

# Theory curve must match the normalisation the model was trained against.
# G0_NORM=True  → G_norm(τ) = G(τ)/G(0) = 1/(1+γτ^α)            (v4.5)
# G0_NORM=False → G_norm(τ) = G(τ)/G(τ_ref) = (1+γτ_ref^α)/(1+γτ^α)
if G0_NORM:
    g_theory_r2 = 1.0 / (1.0 + gamma_3d * tau_3d**alpha_3d + eps)
else:
    g_theory_r2 = ((1 + gamma_3d * tau_ref_r2**alpha_3d)
                   / (1 + gamma_3d * tau_3d**alpha_3d + eps))

g_norm_full = infer_ds.g_norm[:Hv, :Wv]                      # (H, W, K)
ss_res = ((g_theory_r2 - g_norm_full)**2).sum(axis=-1)
ss_tot = ((g_norm_full - g_norm_full.mean(axis=-1,keepdims=True))**2).sum(axis=-1)+eps
r2_map = (1.0 - ss_res/ss_tot).astype(np.float32)
r2_map[~cell_mask] = np.nan

r2_c = r2_map[cell_mask]
print(f'R² stats (cell pixels: {cell_mask.sum()})')
print(f'  mean={np.nanmean(r2_c):.3f}  median={np.nanmedian(r2_c):.3f}')
print(f'  p10={np.nanpercentile(r2_c,10):.3f}  p90={np.nanpercentile(r2_c,90):.3f}')
print(f'  R²>0.9: {(r2_c>0.9).mean()*100:.1f}%   '
      f'R²>0.5: {(r2_c>0.5).mean()*100:.1f}%   '
      f'R²<0 : {(r2_c<0).mean()*100:.1f}%')

fig_r2, axes_r2 = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, cmap, title in [
    (axes_r2[0], gamma_pred, 'magma',  f'Model γ  [{VERSION}]'),
    (axes_r2[1], alpha_pred, 'plasma', f'Model α  [{VERSION}]'),
    (axes_r2[2], r2_map,     'RdYlGn', f'R² confidence  mean={np.nanmean(r2_c):.3f}'),
]:
    p1=np.nanpercentile(data,1); p99=np.nanpercentile(data,99)
    vmin,vmax = (0,1) if 'R²' in title else (p1,p99)
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=11);  ax.axis('off')
plt.suptitle(f'R² Confidence Map [{VERSION}]', fontsize=13)
plt.tight_layout();  plt.show()

r2_clean = np.where(np.isnan(r2_map), 0.0, r2_map)
tifffile.imwrite(os.path.join(SAVE_DIR, f'r2_map_{VERSION}_real.tif'), r2_clean.astype(np.float32))
fig_r2.savefig(os.path.join(SAVE_DIR, f'r2_map_{VERSION}_real.png'), dpi=120, bbox_inches='tight')
print(f'R² map saved → {SAVE_DIR}')


In [ ]:
# ── p2-checkerboard: Cross-validation vs Traditional iSCORS ───────────────
# diagonal_resample() → gridA (even diagonal) / gridB (odd diagonal)
# Both: (T, H//2, W//2), same physics, independent noise.
#   gridA → traditional curve-fit per pixel  (quasi-GT)
#   gridB → model inference                  (model prediction)
import sys, numpy as np, torch, matplotlib.pyplot as plt, os, tqdm
from scipy.optimize import curve_fit
from scipy.stats import pearsonr, spearmanr
import torch.nn.functional as F_

for _k in list(sys.modules.keys()):
    if _k in ('datasets','models') or _k.startswith(('datasets.','models.')):
        del sys.modules[_k]

from utils.sn2n_sampling         import diagonal_resample
from utils.traditional_iscors    import compute_g_empirical_map
from datasets.phys_recon_dataset import PhysReconDataset
from models.pissl_tau_encoder    import PISSLTauEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gridA, gridB = diagonal_resample(video_proc)
print(f'video_proc {video_proc.shape} → gridA {gridA.shape}  gridB {gridB.shape}')

tau_arr_cb = np.array(RECON_TAUS, dtype=np.float32)
tau_ref_cb = float(RECON_TAUS[0]);  eps = 1e-10

# ── Grid A: traditional curve-fit (g0_norm = MATLAB nor_1) ───────────────
# Normalise by G(0)=CV² to match MATLAB 'nor_1': G_norm(τ)=G(τ)/G(0)=1/(1+γτ^α).
# Previously divided by G(τ_ref=1) → shape_only formula; different scale from GT.
print('Computing G_empirical on gridA ...')
gA_map, cellA = compute_g_empirical_map(gridA, RECON_TAUS, min_cv=0.005)

mean_A   = gridA.mean(axis=0)                                # (H/2, W/2)
delta_A  = gridA - mean_A
g_zero_A = (delta_A**2).mean(axis=0) / (mean_A**2 + eps)    # CV² = G(0)
gA_norm  = gA_map / (g_zero_A[:,:,None] + eps)              # G(τ)/G(0)

def _g_norm_cb(tau, gamma, alpha):
    return 1.0 / (1.0 + gamma * tau**alpha)                 # matches MATLAB nor_1

HA,WA = cellA.shape
gamma_trad = np.full((HA,WA), np.nan, dtype=np.float32)
alpha_trad = np.full((HA,WA), np.nan, dtype=np.float32)
r2_trad    = np.zeros((HA,WA), dtype=np.float32)
ys,xs = np.where(cellA)
print(f'Fitting {len(ys)} cell pixels (gridA, g0_norm) ...')
for y,x in tqdm.tqdm(zip(ys,xs), total=len(ys), desc='gridA fit'):
    gp = gA_norm[y,x]
    try:
        po,_ = curve_fit(_g_norm_cb, tau_arr_cb, gp,
                         p0=[0.05,0.8], bounds=([1e-4,0.1],[10.0,2.0]), maxfev=500)
        gamma_trad[y,x]=po[0]; alpha_trad[y,x]=po[1]
        yp=_g_norm_cb(tau_arr_cb,*po)
        r2_trad[y,x]=1-(((gp-yp)**2).sum())/(((gp-gp.mean())**2).sum()+eps)
    except Exception:
        pass
print(f'  γ mean={np.nanmean(gamma_trad):.4f}  α mean={np.nanmean(alpha_trad):.4f}  R² mean={np.nanmean(r2_trad[cellA]):.3f}')

# ── Grid B: model inference ────────────────────────────────────────────────
print('\nModel inference on gridB ...')
cb_ds = PhysReconDataset(gridB, recon_taus=RECON_TAUS, mode='eval')
cb_model = PISSLTauEncoder(recon_taus=RECON_TAUS, predict_amplitude=False).to(device)
cb_model.load_state_dict(torch.load(REAL_CKPT, map_location=device))
cb_model.eval()

ci = cb_ds[0]; K_cb,H_cb,W_cb = ci.shape
ph=(8-H_cb%8)%8; pw=(8-W_cb%8)%8
if ph or pw: ci = F_.pad(ci,(0,pw,0,ph))
with torch.no_grad(): co = cb_model(ci.unsqueeze(0).to(device))
g_cb = co[0,0].cpu().numpy()[:H_cb,:W_cb]
a_cb = co[0,1].cpu().numpy()[:H_cb,:W_cb]
cellB = cb_ds.cell_mask

# ── Comparison ────────────────────────────────────────────────────────────
HH=min(HA,H_cb); WW=min(WA,W_cb)
both = cellA[:HH,:WW] & cellB[:HH,:WW]
gt_g=gamma_trad[:HH,:WW][both]; md_g=g_cb[:HH,:WW][both]
gt_a=alpha_trad[:HH,:WW][both]; md_a=a_cb[:HH,:WW][both]
# Remove NaN from trad fit failures
ok = np.isfinite(gt_g) & np.isfinite(gt_a)
gt_g,md_g,gt_a,md_a = gt_g[ok],md_g[ok],gt_a[ok],md_a[ok]

pg,_=pearsonr(gt_g,md_g); sg=spearmanr(gt_g,md_g).statistic; mg=np.abs(md_g-gt_g).mean()
pa,_=pearsonr(gt_a,md_a); sa=spearmanr(gt_a,md_a).statistic; ma=np.abs(md_a-gt_a).mean()
print(f'\n=== Checkerboard Cross-validation [{VERSION}] ===')
print(f'  Cell pixels: {ok.sum()}')
print(f'  γ  Pearson={pg:.3f}  Spearman={sg:.3f}  MAE={mg:.4f}')
print(f'  α  Pearson={pa:.3f}  Spearman={sa:.3f}  MAE={ma:.4f}')

# 2×3 maps
fig_cb, axes_cb = plt.subplots(2,3,figsize=(15,9))
for row,(trd,mdl,cell,lbl,cmap,mae_v) in enumerate([
    (gamma_trad[:HH,:WW],g_cb[:HH,:WW],both,'γ','magma',mg),
    (alpha_trad[:HH,:WW],a_cb[:HH,:WW],both,'α','plasma',ma),
]):
    for col,(arr,ttl) in enumerate([
        (np.where(cell,trd,np.nan), f'{lbl} trad gridA (quasi-GT)'),
        (np.where(cell,mdl,np.nan), f'{lbl} model gridB'),
        (np.where(cell,np.abs(mdl-trd),np.nan),f'|Δ{lbl}|  MAE={mae_v:.4f}'),
    ]):
        p1=np.nanpercentile(arr,1); p99=np.nanpercentile(arr,99)
        im=axes_cb[row,col].imshow(arr,cmap='hot' if col==2 else cmap,vmin=p1,vmax=p99)
        plt.colorbar(im,ax=axes_cb[row,col],fraction=0.046,pad=0.04)
        axes_cb[row,col].set_title(ttl,fontsize=10); axes_cb[row,col].axis('off')
fig_cb.suptitle(f'Checkerboard CV [{VERSION}]  γ Pearson={pg:.3f}  α Pearson={pa:.3f}',fontsize=13)
plt.tight_layout(); plt.show()

# Scatter
fig_sc,sc_ax=plt.subplots(1,2,figsize=(10,4))
for ax,gt,md,pr,sr,mae,nm in [(sc_ax[0],gt_g,md_g,pg,sg,mg,'γ'),(sc_ax[1],gt_a,md_a,pa,sa,ma,'α')]:
    ax.scatter(gt,md,alpha=0.05,s=1,c='steelblue')
    lim=[min(gt.min(),md.min()),max(gt.max(),md.max())]
    ax.plot(lim,lim,'r--',lw=1,label='y=x'); ax.legend(fontsize=9)
    ax.set_xlabel(f'{nm} trad quasi-GT'); ax.set_ylabel(f'{nm} model')
    ax.set_title(f'{nm}: Pearson={pr:.3f}  Spearman={sr:.3f}  MAE={mae:.4f}')
fig_sc.suptitle(f'Checkerboard Scatter [{VERSION}]',fontsize=13)
plt.tight_layout(); plt.show()

fig_cb.savefig(os.path.join(SAVE_DIR,f'checkerboard_cv_{VERSION}_real.png'),dpi=120,bbox_inches='tight')
fig_sc.savefig(os.path.join(SAVE_DIR,f'checkerboard_scatter_{VERSION}_real.png'),dpi=120,bbox_inches='tight')
print(f'Saved → {SAVE_DIR}')


In [ ]:
# ── p2-gnorm: G_norm Curve Analysis ──────────────────────────────────────
# Empirical G_norm mean±σ vs model theory vs synthetic reference curves
import numpy as np, matplotlib.pyplot as plt, os

tau_arr_gn = np.array(RECON_TAUS, dtype=np.float32)
tau_ref_gn = float(RECON_TAUS[0]);  eps=1e-10

g_full = infer_ds.g_norm[:Hv,:Wv]                            # (H, W, K)
cell   = cell_mask
g_cell = g_full[cell]                                        # (N_cell, K)
g_mean_real = g_cell.mean(axis=0)
g_std_real  = g_cell.std(axis=0)

# Model mean prediction → theory curve
g_m  = np.nanmean(gamma_pred[cell])
a_m  = np.nanmean(alpha_pred[cell])
g_theory_mean = ((1+g_m*tau_ref_gn**a_m)/(1+g_m*tau_arr_gn**a_m+eps))

# Synthetic reference curves
synths = [
    (0.10, 1.0, 'synth cell body  γ=0.10 α=1.0',  'g--'),
    (0.50, 1.5, 'synth fast spot  γ=0.50 α=1.5',  'r--'),
    (0.05, 0.5, 'synth slow spot  γ=0.05 α=0.5',  'm--'),
]

fig_gn, axes_gn = plt.subplots(1,2,figsize=(14,5))
for ax, show_synth in zip(axes_gn, [False, True]):
    ax.fill_between(tau_arr_gn, g_mean_real-g_std_real, g_mean_real+g_std_real,
                    alpha=0.2, label='real ±1σ')
    ax.plot(tau_arr_gn, g_mean_real, 'o-b', label='real mean', lw=1.5)
    if not show_synth:
        ax.plot(tau_arr_gn, g_theory_mean, 'r--',
                label=f'theory γ={g_m:.3f} α={a_m:.3f} (model mean)')
    else:
        for gv,av,lbl,fmt in synths:
            gt = (1+gv*tau_ref_gn**av)/(1+gv*tau_arr_gn**av)
            ax.plot(tau_arr_gn, gt, fmt, label=lbl)
    ax.axhline(1.0,color='gray',lw=0.8,ls=':')
    ax.set_xscale('log');  ax.set_xlabel('τ (lag)');  ax.set_ylabel('G_norm(τ)')
    ax.set_title('Real vs Theory' if not show_synth else 'Real vs Synthetic references')
    ax.legend(fontsize=8);  ax.grid(alpha=0.3)
fig_gn.suptitle(f'G_norm Curve Analysis [{VERSION}]  (real cell, RECON_TAUS={RECON_TAUS})',fontsize=12)
plt.tight_layout();  plt.show()
fig_gn.savefig(os.path.join(SAVE_DIR,f'gnorm_stats_{VERSION}_real.png'),dpi=120,bbox_inches='tight')
print(f'Saved → {SAVE_DIR}')


In [ ]:
# ── p2-download: ZIP all results and download ─────────────────────────────
import zipfile, os, glob
from google.colab import files

zip_name = f'phase2_results_{VERSION}.zip'
patterns = [
    os.path.join(SAVE_DIR, f'*{VERSION}*real*'),
    os.path.join(SAVE_DIR, f'*{VERSION}*'),
    os.path.join(SAVE_DIR, 'gnorm_ch*_tau*.tif'),
]
collected = []
for p in patterns:
    collected.extend(glob.glob(p))
collected = sorted(set(collected))

print(f'Files to zip ({len(collected)}):')
for p in collected:
    print(f'  {os.path.basename(p):50s}  {os.path.getsize(p)/1024:.1f} KB')

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in collected:
        zf.write(p, os.path.basename(p))

print(f'\n{zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)
